In [18]:
from dotenv import load_dotenv
import os

# Get the current working directory
current_directory = os.getcwd()

# Construct path to the .env.example assuming it's in the parent directory of the script's execution directory
env_path = os.path.join(current_directory, '..','..','..','..', '.env.example')

# Load the environment variables from .env.example
load_dotenv(dotenv_path=env_path)

True

In [19]:
#OPENAI_API_KEY = "sk"
from dotenv import load_dotenv

load_dotenv("./../../../.env")
import os
#os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
OPENAI_API_KEY =os.environ.get("OPENAI_API_KEY", None)
#print(OPENAI_API_KEY)

In [20]:
!pip install textgrad # you might need to restart the notebook after installing textgrad

from textgrad.engine import get_engine
from textgrad import Variable
from textgrad.optimizer import TextualGradientDescent
from textgrad.loss import TextLoss
from dotenv import load_dotenv
load_dotenv()

1338.87s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


False

## Introduction: Variable

Variables in TextGrad are the metaphorical equivalent of tensors in PyTorch. They are the primary data structure that you will interact with when using TextGrad.

Variables keep track of gradients and manage the data.

Variables require two arguments (and there is an optional third one):

1. `data`: The data that the variable will hold
2. `role_description`: A description of the role of the variable in the computation graph
3. `requires_grad`: (optional) A boolean flag that indicates whether the variable requires gradients

In [23]:
instruction= "Mick pays his teacher $800 for 40 lessons worth 2 hours each. If this will be all he is going to pay for his lessons, how much did he receive?",
output ="He received 80 hours of lessons.",

In [24]:
x = Variable(instruction, role_description="The instruction I need a prompt for", requires_grad=True)

In [25]:
x.gradients

set()

## Introduction: Engine

When we talk about the engine in TextGrad, we are referring to an LLM. The engine is an abstraction we use to interact with the model.

In [26]:
engine = get_engine("gpt-3.5-turbo")

This object behaves like you would expect an LLM to behave: You can sample generation from the engine using the `generate` method.

In [27]:
engine.generate("Hello how are you?")

"Hello! I'm here and ready to assist you. How can I help you today?"

## Introduction: Loss

Again, Loss in TextGrad is the metaphorical equivalent of loss in PyTorch. We use Losses in different form in TextGrad but for now we will focus on a simple TextLoss. TextLoss is going to evaluate the loss wrt a string.

In [28]:
system_prompt = Variable("Give me a prompt to answer this question", role_description="The system prompt")
loss = TextLoss(system_prompt, engine=engine)

In [29]:
system_prompt

Variable(value=Give me a prompt to answer this question, role=The system prompt, grads=)

## Introduction: Optimizer

Keeping on the analogy with PyTorch, the optimizer in TextGrad is the object that will update the parameters of the model. In this case, the parameters are the variables that have `requires_grad` set to `True`.

**NOTE** This is a text optimizer! It will do all operations with text!

In [30]:
optimizer = TextualGradientDescent(parameters=[x], engine=engine)


## Putting it all together

We can now put all the pieces together. We have a variable, an engine, a loss, and an optimizer. We can now run a single optimization step.

In [31]:
l = loss(x)
l.backward(engine)
optimizer.step()

In [32]:
x

Variable(value=Provide a specific instruction guiding the language model to generate a prompt that calculates the total amount of money Mick received for all his lessons., role=The instruction I need a prompt for, grads=Here is a conversation:

<CONVERSATION><LM_SYSTEM_PROMPT> Give me a prompt to answer this question </LM_SYSTEM_PROMPT>

<LM_INPUT> ('Mick pays his teacher $800 for 40 lessons worth 2 hours each. If this will be all he is going to pay for his lessons, how much did he receive?',) </LM_INPUT>

<LM_OUTPUT> What is the total amount of money Mick received for each lesson he attended? </LM_OUTPUT>

</CONVERSATION>

This conversation is potentially part of a larger system. The output is used as response from the language model

Here is the feedback we got for The instruction I need a prompt for in the conversation:

<FEEDBACK>Since the output prompt generated by the variable is asking for the total amount of money Mick received for each lesson he attended, the current prompt pr

In [33]:
x.value

'Provide a specific instruction guiding the language model to generate a prompt that calculates the total amount of money Mick received for all his lessons.'

In [21]:
optimizer.

While here it is not going to be useful, we can also do multiple optimization steps in a loop! Do not forget to reset the gradients after each step!

In [34]:
optimizer.zero_grad()

# Prompt optimization, minimal example

In [89]:
!pip install wget


8380.54s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9656 sha256=3f5839967bcd98e43a42337a0e8f73ac32d9955e2951dc1f4eee986586da6012
  Stored in directory: /Users/abishekchiffon/Library/Caches/pip/wheels/40/b3/0f/a40dbd1c6861731779f62cc4babcb234387e11d697df70ee97
Successfully built wget


In [138]:
instruction = "What year was the Yamato Battleship built?"
output = "The Yamato Battleship was built in 1941."

In [159]:
import textgrad as tg
from textgrad.tasks import load_task
import wget

llm_engine = tg.get_engine("gpt-3.5-turbo")
#tg.set_backward_engine("gpt-4o")

#_, val_set, _, eval_fn = load_task("BBH_object_counting", llm_engine)
#question_str, answer_str = val_set[0]
question = tg.Variable(instruction, role_description="Prompt engineer this question to the LLM", requires_grad=True)
answer = tg.Variable("This will have the exact answer for the question", role_description="answer to the question", requires_grad=False)

In [154]:
system_prompt = tg.Variable("You are a concise LLM. Think step by step.",
                            requires_grad=True,
                            role_description="system prompt to guide the LLM's reasoning strategy for accurate responses")

model = tg.BlackboxLLM(llm_engine, system_prompt=system_prompt)
optimizer = tg.TGD(parameters=list(model.parameters()))

prediction = model(question)
prediction

Variable(value=The Yamato Battleship was built in 1941., role=response from the language model, grads=)

In [156]:
evaluation_instruction = (f"Here's a question: {question}. " 
                           "Create a prompt that will guide the LLM to answer the question. "
                           "be smart, logical, and very critical. "
                           "Just provide concise feedback.")
                            

# TextLoss is a natural-language specified loss function that describes 
# how we want to evaluate the reasoning.
loss_fn = tg.TextLoss(evaluation_instruction)
loss = loss_fn(answer)
#loss = eval_sample(iprediction, ground_truth_answer=answer))

In [157]:
loss.backward()
optimizer.step()
prediction = model(question)
prediction

Variable(value=The Yamato Battleship was built in 1941. It was commissioned into the Imperial Japanese Navy in December 1941., role=response from the language model, grads=)

Variable(value=The Yamato Battleship was built in 1941. It was one of the largest battleships ever constructed and served in the Imperial Japanese Navy during World War II., role=response from the language model, grads=)

In [146]:
print(question)

What year was the Yamato Battleship built?


In [147]:
print(model.system_prompt.value)

You are a precise and thorough LLM. Carefully consider all relevant information, verify facts, and provide clear, accurate, and well-structured responses. Think step by step.


In [110]:
print(model.parameters())

[Variable(value=You are a precise and thorough LLM. Carefully consider all relevant information, verify facts, and evaluate multiple perspectives before forming your response. Provide clear, accurate, and well-structured answers, ensuring they are concise and easy to understand. Think step by step., role=system prompt to guide the LLM's reasoning strategy for accurate responses, grads=)]
